# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ranahassan9/Machine-Learning-FlyRank-AI/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

We start with two reality checks before encoding a rule.

Signal check A (flag-linked to FlyRank refresh logic): staleness (`days_since_last_update`) vs decline rate.
- Hypothesis: older pages decline more often.
- If this is not true, a "refresh old pages" rule is weak.

Signal check B (flag-linked to CTR-fix logic): low CTR + weak rank among visible pages.
- Hypothesis: pages with poor CTR and page-2/3 positions are better refresh candidates.
- If this is not true, a CTR-focused rule is weak.

Rule we encode:
- Score pages higher when they are old, have strong impressions, low CTR, and weak rank.
- Add an AI-traffic protection penalty so we avoid aggressively editing pages with live AI referrals.

Reason code policy (one reason code per row):
- `STALE_AND_VISIBLE` when staleness is the stronger driver.
- `LOW_CTR_AND_WEAK_RANK` when ranking/snippet weakness is stronger.

Action label:
- `REFRESH_CANDIDATE` for high-priority rows.
- `MONITOR` otherwise.


In [1]:
import pandas as pd
import numpy as np

# Load the Week 4 starter data.
url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Build the decline label only for signal validation (never as a rule input).
df["is_declining_label"] = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)

# ---------- Signal check A: staleness vs decline ----------
# Bucket by days since update to test whether older pages decline more.
staleness_bins = [-1, 30, 90, 180, 365, np.inf]
staleness_labels = ["0-30d", "31-90d", "91-180d", "181-365d", "365d+"]
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"].fillna(df["days_since_last_update"].median()),
    bins=staleness_bins,
    labels=staleness_labels,
)

staleness_table = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "count"),
          decline_rate=("is_declining_label", "mean"),
          median_impressions_90d=("impressions_90d", "median")
      )
      .reset_index()
)
staleness_table["decline_rate"] = staleness_table["decline_rate"].round(4)

print("Signal A: Staleness vs decline")
print(staleness_table.to_string(index=False))

# Simple verdict rule: monotonic rise from newest to oldest buckets => CONFIRMED.
staleness_rates = staleness_table["decline_rate"].tolist()
staleness_verdict = "CONFIRMED" if staleness_rates == sorted(staleness_rates) else "MIXED"
print("Verdict A:", staleness_verdict)

# ---------- Signal check B: CTR-position quality vs decline ----------
# Use visible pages only so CTR/rank has enough support.
visible = df[df["impressions_90d"] >= 500].copy()

# avg_position == 0 means no position data, so remove for this specific test.
visible = visible[visible["avg_position"] > 0].copy()

# Build four buckets that combine CTR quality and ranking zone.
visible["ctr_bucket"] = np.where(visible["ctr"] < 0.03, "low_ctr", "ok_ctr")
visible["rank_bucket"] = np.where(visible["avg_position"] >= 10, "rank_10plus", "rank_top10")
visible["ctr_rank_bucket"] = visible["ctr_bucket"] + "__" + visible["rank_bucket"]

ctr_rank_table = (
    visible.groupby("ctr_rank_bucket")
           .agg(
               n=("content_id", "count"),
               decline_rate=("is_declining_label", "mean"),
               median_impressions_90d=("impressions_90d", "median")
           )
           .reset_index()
           .sort_values("decline_rate", ascending=False)
)
ctr_rank_table["decline_rate"] = ctr_rank_table["decline_rate"].round(4)

print("\nSignal B: CTR-position bucket vs decline (visible pages only)")
print(ctr_rank_table.to_string(index=False))

# Compare worst bucket to best bucket for a readable verdict.
worst = ctr_rank_table.iloc[0]
best = ctr_rank_table.iloc[-1]
ctr_rank_verdict = "CONFIRMED" if worst["decline_rate"] > best["decline_rate"] else "OPPOSITE"
print("Verdict B:", ctr_rank_verdict)

# Keep the verdicts ready for the notebook narrative.
signal_verdicts = {
    "staleness_signal": staleness_verdict,
    "ctr_rank_signal": ctr_rank_verdict,
}
print("\nSignal verdict summary:", signal_verdicts)


Signal A: Staleness vs decline
staleness_bucket     n  decline_rate  median_impressions_90d
           0-30d 20480        0.5114                   470.0
          31-90d   175        0.5886                   510.0
         91-180d  9171        0.6111                  1692.0
        181-365d   169        0.4675                    16.0
           365d+     5        0.6000                     2.0
Verdict A: MIXED

Signal B: CTR-position bucket vs decline (visible pages only)
     ctr_rank_bucket    n  decline_rate  median_impressions_90d
 low_ctr__rank_top10  606        0.8185                  1033.5
 ok_ctr__rank_10plus 7068        0.6034                  2883.0
low_ctr__rank_10plus 2178        0.5927                   996.0
  ok_ctr__rank_top10 6874        0.5687                  5098.0
Verdict B: CONFIRMED

Signal verdict summary: {'staleness_signal': 'MIXED', 'ctr_rank_signal': 'CONFIRMED'}


## 2. Build the ranked queue (writes the CSV)

This cell turns the validated signals into one transparent baseline rule.

Design choices:
- Inputs are pre-decision fields only (no `trend_direction`, no `trend_pct`).
- One score per row.
- One reason code per row.
- One action label per row.
- Writes `work/outputs/baseline_action_score.csv`.


In [ ]:
from pathlib import Path

work_df = df.copy()

# Safe imputation for fields used in the baseline score.
work_df["days_since_last_update"] = work_df["days_since_last_update"].fillna(work_df["days_since_last_update"].median())
work_df["impressions_90d"] = work_df["impressions_90d"].fillna(0)
work_df["ctr"] = work_df["ctr"].fillna(work_df["ctr"].median())
work_df["avg_position"] = work_df["avg_position"].replace(0, np.nan).fillna(work_df["avg_position"].median())
work_df["ai_sessions_90d"] = work_df["ai_sessions_90d"].fillna(0)

# Build normalized signal components in [0,1] to keep the score interpretable.
work_df["staleness_component"] = (work_df["days_since_last_update"] / 365.0).clip(0, 1)
work_df["visibility_component"] = (np.log1p(work_df["impressions_90d"]) / np.log1p(work_df["impressions_90d"].max())).clip(0, 1)
work_df["ctr_problem_component"] = (1 - (work_df["ctr"] / 0.1)).clip(0, 1)
work_df["rank_problem_component"] = ((work_df["avg_position"] - 1) / 30).clip(0, 1)
work_df["ai_protection_component"] = (work_df["ai_sessions_90d"] > 0).astype(int)

# One transparent baseline score: weighted risk minus AI protection penalty.
work_df["baseline_score"] = (
    0.35 * work_df["staleness_component"]
    + 0.35 * work_df["visibility_component"]
    + 0.20 * work_df["ctr_problem_component"]
    + 0.10 * work_df["rank_problem_component"]
    - 0.20 * work_df["ai_protection_component"]
)

# One reason code per row: whichever signal family dominates.
work_df["reason_code"] = np.where(
    work_df["staleness_component"] >= (work_df["ctr_problem_component"] + work_df["rank_problem_component"]) / 2,
    "STALE_AND_VISIBLE",
    "LOW_CTR_AND_WEAK_RANK",
)

# Visibility gate to avoid pushing near-zero-demand pages to the top of the queue.
min_visibility_for_refresh = 500
refresh_cutoff = 0.55
work_df["action_label"] = np.where(
    (work_df["baseline_score"] >= refresh_cutoff) & (work_df["impressions_90d"] >= min_visibility_for_refresh),
    "REFRESH_CANDIDATE",
    "MONITOR",
)

# Build ranked queue (highest score first).
queue_cols = [
    "content_id", "client_id", "baseline_score", "action_label", "reason_code",
    "days_since_last_update", "impressions_90d", "ctr", "avg_position", "ai_sessions_90d",
]
ranked_queue = work_df[queue_cols].sort_values("baseline_score", ascending=False).reset_index(drop=True)
ranked_queue["rank"] = ranked_queue.index + 1

# Write output CSV required by Week 4.
out_path = Path("work/outputs/baseline_action_score.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
ranked_queue.to_csv(out_path, index=False)

print("Wrote:", out_path)
print("Rows written:", len(ranked_queue))
print("Refresh candidates:", int((ranked_queue["action_label"] == "REFRESH_CANDIDATE").sum()))
print("\nTop 5 rows:")
print(ranked_queue.head(5).to_string(index=False))


Wrote: work\outputs\baseline_action_score.csv
Rows written: 30000

Top 5 rows:
          content_id         client_id  baseline_score      action_label           reason_code  days_since_last_update  impressions_90d  ctr  avg_position  ai_sessions_90d  rank
content_6476d1d8c050 client_19581e27de        0.801706 REFRESH_CANDIDATE LOW_CTR_AND_WEAK_RANK                     313              304  0.0          67.8                0     1
content_d25a099b3726 client_19581e27de        0.785203 REFRESH_CANDIDATE LOW_CTR_AND_WEAK_RANK                     305              202  0.0          64.5                0     2
content_7a888d3d99c8 client_19581e27de        0.779741 REFRESH_CANDIDATE LOW_CTR_AND_WEAK_RANK                     313               95  0.0          67.6                0     3
content_f6fdf87348f6 client_4ec9599fc2        0.770875 REFRESH_CANDIDATE     STALE_AND_VISIBLE                     373                2  0.0          32.5                0     4
content_8d56efff1e71 client_4ec

## 3. Top-10 review

We read the top 10 with a skeptic's eye.

For each row we provide:
- action label
- reason code
- why it appears in the top 10
- what would make this pick wrong

A good baseline review includes uncertainty, not just confidence.


In [3]:
top10 = ranked_queue.head(10).copy()

# Build a concise human review line for each top pick.
def make_review_line(row):
    action = row["action_label"]
    reason = row["reason_code"]

    why = (
        f"high score={row['baseline_score']:.3f}, age={int(row['days_since_last_update'])}d, "
        f"impr_90d={int(row['impressions_90d'])}, ctr={row['ctr']:.4f}, pos={row['avg_position']:.2f}"
    )

    wrong_if = (
        "query seasonality changed, or the page is intentionally stable and conversion-focused, "
        "or AI-protected traffic should block refresh edits"
    )

    return f"{action} | {reason} | why: {why} | could be wrong if: {wrong_if}"

review_lines = [make_review_line(row) for _, row in top10.iterrows()]

print("Top-10 skeptical review:")
for i, line in enumerate(review_lines, start=1):
    print(f"{i}. {line}")

# Also keep a structured top-10 table for quick grading.
review_table = top10[[
    "rank", "content_id", "action_label", "reason_code", "baseline_score",
    "days_since_last_update", "impressions_90d", "ctr", "avg_position", "ai_sessions_90d"
]].copy()

print("\nTop-10 table:")
print(review_table.to_string(index=False))


Top-10 skeptical review:
1. REFRESH_CANDIDATE | LOW_CTR_AND_WEAK_RANK | why: high score=0.802, age=313d, impr_90d=304, ctr=0.0000, pos=67.80 | could be wrong if: query seasonality changed, or the page is intentionally stable and conversion-focused, or AI-protected traffic should block refresh edits
2. REFRESH_CANDIDATE | LOW_CTR_AND_WEAK_RANK | why: high score=0.785, age=305d, impr_90d=202, ctr=0.0000, pos=64.50 | could be wrong if: query seasonality changed, or the page is intentionally stable and conversion-focused, or AI-protected traffic should block refresh edits
3. REFRESH_CANDIDATE | LOW_CTR_AND_WEAK_RANK | why: high score=0.780, age=313d, impr_90d=95, ctr=0.0000, pos=67.60 | could be wrong if: query seasonality changed, or the page is intentionally stable and conversion-focused, or AI-protected traffic should block refresh edits
4. REFRESH_CANDIDATE | STALE_AND_VISIBLE | why: high score=0.771, age=373d, impr_90d=2, ctr=0.0000, pos=32.50 | could be wrong if: query seasonality ch

## 4. Weak picks + leakage check

This final check does two things:
1. Identifies weak picks from the top 10 (cases where score is high but evidence is thin).
2. Confirms leakage safety (no label-derived columns or future-window-only fields in the score inputs).

If weak picks exist, that is healthy. It shows we are auditing the rule honestly.


In [4]:
# Identify weak picks among top-10 by simple risk heuristics.
# Weak means: low visibility, high AI risk, or borderline score.
weak_top10 = review_table[
    (review_table["impressions_90d"] < 800)
    | (review_table["ai_sessions_90d"] > 0)
    | (review_table["baseline_score"] < 0.60)
].copy()

print("Weak picks in top-10:")
if len(weak_top10) == 0:
    print("None under current thresholds.")
else:
    print(weak_top10.to_string(index=False))

# Leakage check: make sure score did NOT use label-derived or future-only columns.
score_input_columns = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position",
    "ai_sessions_90d",
]

forbidden_columns = [
    "trend_direction",  # directly tied to label definition
    "trend_pct",        # directly tied to label definition
    "is_declining_label" # target label
]

overlap_forbidden = sorted(set(score_input_columns) & set(forbidden_columns))

# Future-window caution check (none used in score): last_30d and prev_30d columns are excluded.
future_window_like = [c for c in score_input_columns if ("last_30d" in c or "prev_30d" in c)]

print("\nLeakage check results:")
print("Score input columns:", score_input_columns)
print("Forbidden overlap:", overlap_forbidden)
print("Future-window overlap:", future_window_like)
print("Leakage verdict:", "PASS" if len(overlap_forbidden) == 0 and len(future_window_like) == 0 else "FAIL")

# Optional sanity metric for baseline context: precision@10 against decline label.
# This is not the final model metric, only a baseline reference point.
top10_idx = ranked_queue.head(10).index
ranked_with_label = ranked_queue.merge(df[["content_id", "is_declining_label"]], on="content_id", how="left")
precision_at_10 = ranked_with_label.head(10)["is_declining_label"].mean()
base_rate = df["is_declining_label"].mean()
print("\nBaseline reference metrics:")
print(f"Precision@10: {precision_at_10:.4f}")
print(f"Base rate (decline prevalence): {base_rate:.4f}")


Weak picks in top-10:
 rank           content_id      action_label           reason_code  baseline_score  days_since_last_update  impressions_90d  ctr  avg_position  ai_sessions_90d
    1 content_6476d1d8c050 REFRESH_CANDIDATE LOW_CTR_AND_WEAK_RANK        0.801706                     313              304  0.0          67.8                0
    2 content_d25a099b3726 REFRESH_CANDIDATE LOW_CTR_AND_WEAK_RANK        0.785203                     305              202  0.0          64.5                0
    3 content_7a888d3d99c8 REFRESH_CANDIDATE LOW_CTR_AND_WEAK_RANK        0.779741                     313               95  0.0          67.6                0
    4 content_f6fdf87348f6 REFRESH_CANDIDATE     STALE_AND_VISIBLE        0.770875                     373                2  0.0          32.5                0
    5 content_8d56efff1e71 REFRESH_CANDIDATE     STALE_AND_VISIBLE        0.763171                     372                1  0.0          35.0                0
    6 content_afd2

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.